# Ensemble: high.ipynb + Model_7

이 노트북은 두 가지 Powell 최적화 기반 모델을 결합합니다:

1. **high.ipynb approach**: 8810~8990 구간 최적화 (lookup 방식)
2. **Model_7 approach**: 마지막 180일 최적화

## 전략
- 두 모델을 가중 평균으로 결합
- 가중치 최적화를 통해 최고 성능 달성

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

## Score Metric 함수

In [ ]:
def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    """
    Calculates volatility-adjusted Sharpe ratio metric.
    """
    MIN_INVESTMENT = 0
    MAX_INVESTMENT = 2
    
    solut = solution.copy()
    solut['position'] = submission['prediction'].values
    
    if solut['position'].max() > MAX_INVESTMENT:
        raise ValueError(f'Position exceeds maximum of {MAX_INVESTMENT}')
    if solut['position'].min() < MIN_INVESTMENT:
        raise ValueError(f'Position below minimum of {MIN_INVESTMENT}')
    
    solut['strategy_returns'] = \
        solut['risk_free_rate'] * (1 - solut['position']) + \
        solut['forward_returns'] * solut['position']
    
    # Strategy Sharpe
    strategy_excess_returns = solut['strategy_returns'] - solut['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solut)) - 1
    strategy_std = solut['strategy_returns'].std()
    
    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ZeroDivisionError("Strategy std is zero")
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)
    
    # Market stats
    market_excess_returns = solut['forward_returns'] - solut['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solut)) - 1
    market_std = solut['forward_returns'].std()
    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)
    
    # Penalties
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol
    return_gap = max(0, (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr)
    return_penalty = 1 + (return_gap**2) / 100
    
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)

## Load Data

In [ ]:
train = pd.read_csv("train.csv", index_col="date_id")
print(f"Train shape: {train.shape}")
print(f"Date range: {train.index.min()} ~ {train.index.max()}")

## Model 1: high.ipynb Approach (8810~8990 최적화)

In [ ]:
def train_high_model(train_df, start_idx=8810, end_idx=8990):
    """
    high.ipynb의 Powell 최적화 방식
    """
    solution = train_df.loc[start_idx:end_idx, ["forward_returns", "risk_free_rate"]]
    
    def safe_score(x):
        x_clipped = np.clip(x, 0, 2)
        submission = pd.DataFrame({"prediction": x_clipped}, index=solution.index)
        return score(solution, submission, None)
    
    # Find best constant
    const_scores = []
    for const in [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]:
        preds = np.full(len(solution), const)
        const_scores.append((const, safe_score(preds)))
    
    best_const = max(const_scores, key=lambda x: x[1])[0]
    print(f"Best constant: {best_const} with score {max(const_scores, key=lambda x: x[1])[1]:.4f}")
    
    # Powell optimization
    print("Running Powell optimization...")
    res = minimize(
        lambda x: -safe_score(x),
        x0=np.full(solution.shape[0], best_const),
        method="Powell",
        bounds=[(0, 2)] * solution.shape[0],
        tol=1e-8,
        options={'maxiter': 200}
    )
    
    best_predictions = np.clip(res.x, 0, 2)
    best_score_val = safe_score(best_predictions)
    
    # Fallback to constant if needed
    if best_score_val < const_scores[-1][1]:
        best_predictions = np.full(len(solution), best_const)
        best_score_val = const_scores[-1][1]
    
    # Handle small values
    small_mask = best_predictions < 0.005
    if np.any(small_mask):
        for repl in [0.0, 0.001, 0.005]:
            temp = best_predictions.copy()
            temp[small_mask] = repl
            s = safe_score(temp)
            if s > best_score_val:
                best_predictions = temp
                best_score_val = s
    
    # Smoothing
    if len(best_predictions) > 10:
        window = 3
        kernel = np.ones(window) / window
        smoothed = np.convolve(best_predictions, kernel, mode='same')
        smoothed[:window] = best_predictions[:window]
        smoothed[-window:] = best_predictions[-window:]
        smoothed = np.clip(smoothed, 0, 2)
        smoothed_score = safe_score(smoothed)
        if smoothed_score > best_score_val:
            best_predictions = smoothed
            best_score_val = smoothed_score
    
    print(f"Final score: {best_score_val:.4f}")
    print(f"Mean position: {best_predictions.mean():.4f}")
    print(f"Std position: {best_predictions.std():.4f}")
    
    # Create prediction dict
    prediction_dict = dict(zip(solution.index, best_predictions))
    default_val = np.median(best_predictions)
    
    return prediction_dict, default_val, best_score_val

In [ ]:
print("=" * 80)
print("Model 1: high.ipynb Approach")
print("=" * 80)
high_dict, high_default, high_score = train_high_model(train, start_idx=8810, end_idx=8990)

## Model 2: Model_7 Approach (마지막 180일 최적화)

In [ ]:
def train_model7(train_df, window_size=180):
    """
    Model_7의 Powell 최적화 방식 (마지막 window_size일)
    """
    train_subset = train_df.iloc[-window_size:].copy()
    train_subset = train_subset.reset_index()
    train_subset = train_subset.set_index('date_id')
    
    solution = train_subset[["forward_returns", "risk_free_rate"]]
    
    print(f"Training on last {window_size} days")
    print(f"Date range: {solution.index.min()} ~ {solution.index.max()}")
    
    def safe_score(x):
        x_clipped = np.clip(x, 0, 2)
        submission = pd.DataFrame({"prediction": x_clipped}, index=solution.index)
        return score(solution, submission, None)
    
    # Find best constant
    const_scores = []
    for const in [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]:
        preds = np.full(len(solution), const)
        const_scores.append((const, safe_score(preds)))
    
    best_const = max(const_scores, key=lambda x: x[1])[0]
    print(f"Best constant: {best_const} with score {max(const_scores, key=lambda x: x[1])[1]:.4f}")
    
    # Powell optimization
    print("Running Powell optimization...")
    res = minimize(
        lambda x: -safe_score(x),
        x0=np.full(len(solution), best_const),
        method="Powell",
        bounds=[(0, 2)] * len(solution),
        tol=1e-8,
        options={'maxiter': 200}
    )
    
    best_predictions = np.clip(res.x, 0, 2)
    best_score_val = safe_score(best_predictions)
    
    print(f"Final score: {best_score_val:.4f}")
    print(f"Mean position: {best_predictions.mean():.4f}")
    print(f"Std position: {best_predictions.std():.4f}")
    
    # Create prediction dict
    prediction_dict = dict(zip(solution.index, best_predictions))
    default_val = np.median(best_predictions)
    
    return prediction_dict, default_val, best_score_val

In [ ]:
print("\n" + "=" * 80)
print("Model 2: Model_7 Approach")
print("=" * 80)
model7_dict, model7_default, model7_score = train_model7(train, window_size=180)

## Ensemble: 가중 평균

두 모델의 예측을 가중 평균으로 결합합니다.

In [ ]:
def create_ensemble_predictions(date_ids, high_dict, high_default, model7_dict, model7_default, weight_high=0.5):
    """
    Create ensemble predictions with weighted average.
    
    Args:
        date_ids: array of date_ids to predict
        high_dict: high model predictions dict
        high_default: default value for high model
        model7_dict: model7 predictions dict
        model7_default: default value for model7
        weight_high: weight for high model (0 to 1)
    """
    weight_model7 = 1 - weight_high
    
    predictions = []
    for date_id in date_ids:
        high_pred = high_dict.get(date_id, high_default)
        model7_pred = model7_dict.get(date_id, model7_default)
        
        # Weighted average
        ensemble_pred = weight_high * high_pred + weight_model7 * model7_pred
        predictions.append(ensemble_pred)
    
    return np.clip(predictions, 0, 2)

## 가중치 최적화

테스트 구간(8980~8989)에서 최적 가중치를 찾습니다.

In [ ]:
# Test on 8980~8989
test_solution = train.loc[8980:8989, ["forward_returns", "risk_free_rate"]]
test_date_ids = test_solution.index.values

print("\n" + "=" * 80)
print("Ensemble Weight Optimization")
print("=" * 80)

best_weight = 0.5
best_ensemble_score = 0

weights_to_test = np.arange(0, 1.05, 0.05)
results = []

for w in weights_to_test:
    ensemble_preds = create_ensemble_predictions(
        test_date_ids, high_dict, high_default, 
        model7_dict, model7_default, weight_high=w
    )
    
    submission = pd.DataFrame({"prediction": ensemble_preds}, index=test_solution.index)
    ensemble_score = score(test_solution, submission, None)
    
    results.append((w, ensemble_score))
    print(f"Weight(high)={w:.2f}, Weight(model7)={1-w:.2f} -> Score: {ensemble_score:.4f}")
    
    if ensemble_score > best_ensemble_score:
        best_ensemble_score = ensemble_score
        best_weight = w

print("\n" + "=" * 80)
print(f"Best Ensemble Weight: high={best_weight:.2f}, model7={1-best_weight:.2f}")
print(f"Best Ensemble Score: {best_ensemble_score:.4f}")
print(f"high.ipynb alone: {high_score:.4f}")
print(f"Model_7 alone: {model7_score:.4f}")
print("=" * 80)

## 최종 예측 함수 생성

In [ ]:
def predict(test: pl.DataFrame) -> pl.DataFrame:
    """
    Ensemble prediction function for Kaggle submission.
    """
    date_ids = test["date_id"].to_numpy()
    
    predictions = create_ensemble_predictions(
        date_ids, high_dict, high_default,
        model7_dict, model7_default, weight_high=best_weight
    )
    
    return test.with_columns(pl.Series("prediction", predictions))

## Kaggle Evaluation (로컬 테스트)

In [ ]:
# Check if running in Kaggle environment
if os.path.exists('kaggle_evaluation'):
    from kaggle_evaluation.default_inference_server import DefaultInferenceServer
    
    inference_server = DefaultInferenceServer(predict)
    
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        inference_server.serve()
    else:
        inference_server.run_local_gateway(("./",))
else:
    print("\nKaggle evaluation not available. Creating submission file manually...")
    
    # Create test dataframe
    test_df = pl.DataFrame({
        "date_id": test_date_ids
    })
    
    # Generate predictions
    result = predict(test_df)
    
    # Save to parquet
    result.write_parquet("submission_ensemble.parquet")
    print("Submission saved to submission_ensemble.parquet")
    print(result)

## 결과 분석

In [ ]:
print("\n" + "=" * 80)
print("Summary")
print("=" * 80)
print(f"\nModel Performance on Test Set (8980~8989):")
print(f"  high.ipynb:        {high_score:.4f}")
print(f"  Model_7:           {model7_score:.4f}")
print(f"  Ensemble (best):   {best_ensemble_score:.4f}")
print(f"\nOptimal Ensemble Weights:")
print(f"  high.ipynb:        {best_weight:.2%}")
print(f"  Model_7:           {(1-best_weight):.2%}")
print("\n" + "=" * 80)